## Script to initialize the schema and audit tables

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("""
          use catalog novacart_catalog
          """)

In [0]:
spark.sql("""
          CREATE SCHEMA IF NOT EXISTS novacart_catalog.audit
          MANAGED LOCATION 'abfss://novacart-container@novacartdatalake.dfs.core.windows.net/audit'
          """)


spark.sql("""
          CREATE SCHEMA IF NOT EXISTS novacart_catalog.bronze
          MANAGED LOCATION 'abfss://novacart-container@novacartdatalake.dfs.core.windows.net/bronze'
          """)

#### Audit control table setup
This tables stores the watermark for each source tables

It helps the pipeline to remember:
- the last precessed timestamp
- Last processed Primary key at the timestamp
- And how many rows processed


In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS novacart_catalog.audit.ingestion_control_table (
        layer STRING,
        table_name STRING,
        timestamp_col STRING,
        primary_col STRING,
        last_ingested_ts TIMESTAMP,
        last_ingested_pk BIGINT,
        last_run_id STRING,
        rows_written BIGINT,
        run_status STRING,
        updated_at TIMESTAMP
        )
        USING DELTA
        LOCATION 'abfss://novacart-container@novacartdatalake.dfs.core.windows.net/audit/ingestion_control_table'
          """)

#### Source table configuration
This Cell defines which source table will be loaded into bronze and which columns should be used as:
- primary key
- timestamp/watermark column

it also creates a unique id for its run in bronze layer 

In [0]:
tables_config = {
    "orders":{"pk_col":"order_id","ts_col":"updated_at"},
    "products":{"pk_col":"product_id","ts_col":"updated_at"},
    "payments":{"pk_col":"payment_id","ts_col":"processed_at"}
}

##### Unique run id
Create a unique run id for bronze layer ingestion


In [0]:
run_id = str(uuid.uuid4())
print(run_id)

#### configuring Bronze tables and Incremental load loop
- Here we will write a function that will take source table names from the tables_config and creates them in bronze layer as external tables using function "create_bronze_tables" from job_control file.
- Once tables are created in bronze schema, load loop will decide which load will perform on these tbles i.e initial load or incremental load, for each table, loop reads:

    - reads last watermark column value
    - reads source table name
    - filters only new/changed rows 
    - adds bronze audit columns
    - appends or insert the rows into bronze tables




In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from src.utils.job_control import (
    get_last_successful_timestamp,
    upsert_ingestion_control
)


for table_name,cfg in tables_config.items():
    primary_col = cfg['pk_col']
    timestamp_col = cfg['ts_col']
    source_table = f"`novacat-azuresql-connection-lakehouse-federation_catalog`.dbo.{table_name}"

    target_table = f"novacart_catalog.bronze.{table_name}_brz"
    target_table_location = (
        f"abfss://novacart-container@novacartdatalake.dfs.core.windows.net/"
        f"bronze/{table_name}_brz"
                )

    last_ingested_ts,last_ingested_pk = get_last_successful_timestamp(spark,table_name)

    print(f"\n ==== Processing {table_name} ====")
    print(f"last_ingested_ts:{last_ingested_ts}")
    print(f"last_ingested_pk:{last_ingested_pk}")

    source_df = spark.read.table(source_table) \
                     .withColumn(timestamp_col, F.col(timestamp_col).cast("timestamp"))
    
    if last_ingested_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(timestamp_col) > F.lit(last_ingested_ts)) |
            (
                (F.col(timestamp_col) == F.lit(last_ingested_ts)) &
                (F.col(primary_col) > F.lit(int(last_ingested_pk)))
            )
        )

    rows_to_load = (
        rows_to_load
            .withColumn("bronze_ingested_at",F.current_timestamp())
            .withColumn("bronze_run_id",F.lit(run_id))
            .withColumn("bronze_source",F.lit(source_table))

    )

    rows_count = rows_to_load.count()
    print(f"{target_table} rows to load = {rows_count}")

    if rows_count == 0:
        print(f"No new rows to load for {table_name}")
        upsert_ingestion_control(
            spark,
            table_name,
            timestamp_col,
            primary_col,
            last_ingested_ts,
            last_ingested_pk,
            run_id,
            rows_count
        )

        continue

    rows_to_load.write.format("delta").mode("append").option("path",target_table_location).saveAsTable(target_table)

    max_ts = rows_to_load.agg(F.max(timestamp_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk = (
        rows_to_load.filter(F.col(timestamp_col) == F.lit(max_ts))
                        .agg(F.max(primary_col).cast("long").alias("max_pk"))
                        .collect()[0]["max_pk"]
            )
        

    upsert_ingestion_control(
        spark,
        table_name,
        timestamp_col,
        primary_col,
        max_ts,
        max_pk,
        run_id,
        rows_count
    )

    print(f"wrote {rows_count} rows to {target_table}")  



#### Quick Validation

This Finl cell validate the row counts from bronze table and audit table, so that we can identify if the incremental logic workng or not perectly

In [0]:
print("Orders bronze count",spark.sql("""
                                      SELECT count(*) FROM novacart_catalog.bronze.orders_brz
                                      """).collect()[0][0])

print("products bronze count",spark.sql("""
                                      SELECT count(*) FROM novacart_catalog.bronze.products_brz
                                      """).collect()[0][0])

print("payments bronze count",spark.sql("""
                                      SELECT count(*) FROM novacart_catalog.bronze.payments_brz
                                      """).collect()[0][0])

display(spark.sql("""
                 select * from novacart_catalog.audit.ingestion_control_table
                 """).orderBy("table_name"))

